# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading, reviewing, processing, and visualizing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema:
- URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

The dataset consists of clinical, pathological, and biomarker annotations of 77 cancer survivors with second primary colorectal cancer, supporting MSI-H status and anatomical distribution analysis.

In [ ]:
# Install required libraries if needed
!pip install mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata using the metadata property
print("Dataset Name:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("Identifier:", dataset.metadata.identifier)
print("Publication Date:", dataset.metadata.datePublished)


## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

We will inspect the dataset's record sets and their fields, referencing everything by `@id` as required.

In [ ]:
# Get a list of all available record sets by @id
record_sets = dataset.record_sets
print("Record sets found:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# For demonstration, list fields and columns for each record set
for rs in record_sets:
    print(f"\nRecordSet @id: {rs['@id']} (name: {rs.get('name', 'N/A')})")
    if 'field' in rs and isinstance(rs['field'], list):
        print("Fields:")
        for field in rs['field']:
            print(f"  - @id: {field['@id']}, name: {field.get('name', 'N/A')}, dataType: {field.get('dataType', 'N/A')}")
    if 'column' in rs and isinstance(rs['column'], list):
        print("Columns:")
        for col in rs['column']:
            print(f"  - @id: {col['@id']}, name: {col.get('name', 'N/A')}, dataType: {col.get('dataType', 'N/A')}")


## 3. Data Extraction
Load records from each record set using their `@id`. Use the record set and field `@id` references gathered above.

We will load all record sets found, and create a Pandas DataFrame for each.

In [ ]:
# Extract all data using @id
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet @id: {record_set_id}")
        print("Columns:", df.columns.tolist())
        print(df.head(3), "\n")
    else:
        print(f"No records found for RecordSet @id: {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply typical EDA and preprocessing using fields referenced by their `@id`.

- Filter records based on a numeric field (such as age or interval between diagnoses)
- Normalize numeric fields
- Group records by categorical attributes

We'll assume one main record set contains patient-level clinical data, e.g., called `'cr:RecordSet/Patient'` (update based on actual listing above). We reference fields by their exact `@id`.

In [ ]:
# Example: Identify main record set containing patient data
main_record_set_id = None
for rs in record_sets:
    if 'Patient' in rs.get('name','') or 'clinical' in rs.get('name','').lower():
        main_record_set_id = rs['@id']

# If not found, use the first found record set
if not main_record_set_id and len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]

df_main = dataframes.get(main_record_set_id, pd.DataFrame())

# Show column ids and select a numeric field
print("Columns in DF:", df_main.columns.tolist())
numeric_field_id = None
for col in df_main.columns:
    if 'age' in col.lower() or 'interval' in col.lower():
        numeric_field_id = col
        break
if not numeric_field_id:
    # Try to select based on data types if column info is available
    numeric_field_id = df_main.select_dtypes(include='number').columns[0] if len(df_main.select_dtypes(include='number').columns) > 0 else df_main.columns[0]

print("Using numeric field @id for EDA:", numeric_field_id)
threshold = 50

# Filter
filtered_df = df_main[df_main[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (count: {len(filtered_df)}):")
print(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())/filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping by a categorical field
group_field_id = None
for col in df_main.columns:
    if ('sex' in col.lower()) or ('msi' in col.lower()) or ('location' in col.lower()):
        group_field_id = col
        break
if group_field_id:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped.head())

## 5. Visualization
Visualize distributions and relationships between fields. We reference field IDs, not names, as per requirements.

In [ ]:
# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df_main[numeric_field_id], bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot grouped by group_field_id if available
if group_field_id:
    plt.figure(figsize=(8,4))
    sns.boxplot(data=df_main, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()


## 6. Conclusion
This notebook demonstrates FAIR² dataset exploration with `mlcroissant`, referencing all entities via their `@id` fields. Steps include:
- Loading metadata and record sets.
- Extracting and preparing data using record set and field `@id`s.
- Filtering and normalizing numeric fields, grouping by categorical attributes.
- Visualizing distributions and relationships.

**Key takeaways:**
- All schema elements are referenced by `@id`, in alignment with Croissant standards.
- Data processing and visualization steps are practical for clinical and molecular discovery.
- The dataset can be directly used for further modeling or stratification analyses using pandas and mlcroissant.